## 03 — L2 books
Plot BBO/spread; assert sorted sides, aligned arrays, positive quantities.

In [ ]:
import clickhouse_connect, os
client = clickhouse_connect.get_client(
    host=os.environ.get('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.environ.get('CLICKHOUSE_PORT', '8123')),
    username=os.environ.get('CLICKHOUSE_USER', 'default'),
    password=os.environ.get('CLICKHOUSE_PASSWORD', ''),
    database=os.environ.get('CLICKHOUSE_DATABASE', 'market'),
)
print('clickhouse', client.server_version)


In [ ]:
books = client.query("SELECT venue, instrument, bids__price, bids__quantity, asks__price, asks__quantity FROM l2_books FINAL WHERE length(bids__price) > 0 LIMIT 50").result_rows
assert books, 'no book rows'

In [ ]:
for venue, instrument, bp, bq, ap, aq in books:
    assert bp == sorted(bp, reverse=True), 'bids not sorted desc'
    assert ap == sorted(ap), 'asks not sorted asc'
    assert len(bp) == len(bq) and len(ap) == len(aq), 'misaligned arrays'
    assert all(q > 0 for q in bq + aq), 'non-positive quantity'
print('book invariants OK on', len(books), 'rows')

In [ ]:
import matplotlib.pyplot as plt
bp = books[0][2]; ap = books[0][4]
plt.step(range(len(bp)), bp, where='post', label='bids')
plt.step(range(len(ap)), ap, where='post', label='asks')
plt.legend(); plt.title('Top of book'); plt.show()